Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Install independencies

In [ ]:
!pip install anthropic --break-system-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 17.7 MB/s eta 0:00:00


For multi images

In [3]:
import os
import json
import base64
import re
import time
import pandas as pd
from openai import OpenAI
from PIL import Image
from tqdm import tqdm
import io

os.environ["PYTHONIOENCODING"] = "utf-8"
client = userdata.get('GOOGLE_API_KEY')
# ===============================
# 路径配置
# ===============================
test_image_dir = "/content/drive/MyDrive/MyThesis2026/data/cindy/images/test"
test_csv = "/content/drive/MyDrive/MyThesis2026/data/cindy/data/test.csv"
output_json = "/content/drive/MyDrive/GPT4omini_Misogyny_ZeroShot_pred.json"

test_df = pd.read_csv(test_csv)

prompt_text = """You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is misogynistic or not.

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing both the image and the embedded text to determine whether it contains misogynistic content.
Step 2: If the meme contains any negative, insulting, stereotyping, or degrading reference to women, output Misogyny.
Step 3: If the meme does not contain any misogynistic content, output Non_Misogyny.

Output:
Your output should strictly follow the format:
Class labels: Misogyny or Non_Misogyny
Thought: Give your reason here"""

def encode_image(image_path):
    image = Image.open(image_path).convert("RGB")
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG")
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("utf-8")

def call_with_retry(image_data, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}},
                            {"type": "text", "text": prompt_text}
                        ]
                    }
                ],
                max_tokens=300
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                wait = 60 * (attempt + 1)
                print(f"  Rate limit，等待 {wait} 秒后重试...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("超过最大重试次数")

def parse_label(raw):
    m = re.search(r"Class labels?:\**\s*(Misogyny|Non_Misogyny)", raw, re.IGNORECASE)
    if m:
        return m.group(1)
    elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry"]):
        return "Non_Misogyny"
    elif "misogyn" in raw.lower():
        return "Misogyny"
    else:
        return "UNKNOWN"

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining_df = test_df[~test_df["filename"].isin(done_images)]
print(f"剩余待处理: {len(remaining_df)} 张")

for _, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="推理进度"):
    img_name = row["filename"]
    img_path = os.path.join(test_image_dir, img_name)

    if not os.path.exists(img_path):
        print(f"⚠️ 图片不存在: {img_name}")
        continue

    try:
        image_data = encode_image(img_path)
        raw = call_with_retry(image_data)
        label = parse_label(raw)

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw,
            "true_label": int(row["label"])
        })

        print(f"✅ {img_name} -> {label} (真实: {'Misogyny' if row['label']==1 else 'Non_Misogyny'})")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        time.sleep(3)

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e),
            "true_label": int(row["label"])
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        time.sleep(5)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

没有已有结果，从头开始...
剩余待处理: 340 张


推理进度:   0%|          | 0/340 [00:00<?, ?it/s]

✅ 1582.jpg -> Non_Misogyny (真实: Non_Misogyny)


推理进度:   0%|          | 1/340 [00:12<1:10:34, 12.49s/it]

✅ 1305.jpg -> Non_Misogyny (真实: Non_Misogyny)


...

✅ 1091.jpg -> Non_Misogyny (真实: Misogyny)


推理进度: 100%|█████████▉| 339/340 [41:37<00:07,  7.65s/it]

✅ 1103.jpg -> Non_Misogyny (真实: Non_Misogyny)


推理进度: 100%|██████████| 340/340 [41:43<00:00,  7.36s/it]


完成！共 340 条结果已保存
  Misogyny: 58
  Non_Misogyny: 282


Calculate Zero-shot result

In [4]:
import json
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

output_json = "/content/drive/MyDrive/GPT4omini_Misogyny_ZeroShot_pred.json"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

def normalize_pred(x):
    x = str(x).strip().lower()
    if x == "misogyny":
        return "misogyny"
    return "non_misogyny"

def normalize_true(x):
    return "misogyny" if x == 1 else "non_misogyny"

y_true = [normalize_true(p["true_label"]) for p in predictions]
y_pred = [normalize_pred(p["predicted_label"]) for p in predictions]

print("总数:", len(predictions))
print("Unique TRUE labels:", sorted(set(y_true)))
print("Unique PRED labels:", sorted(set(y_pred)))

ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

总数: 340
Unique TRUE labels: ['misogyny', 'non_misogyny']
Unique PRED labels: ['misogyny', 'non_misogyny']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.8000
MP   : 0.8041
MR   : 0.7027
MF1  : 0.7245
WP   : 0.8017
WR   : 0.8000
WF1  : 0.7805
-------------------------------------------------

Confusion Matrix:
[[ 47  57]
 [ 11 225]]

Classification Report:
              precision    recall  f1-score   support

    misogyny       0.81      0.45      0.58       104
non_misogyny       0.80      0.95      0.87       236

    accuracy                           0.80       340
   macro avg       0.80      0.70      0.72       340
weighted avg       0.80      0.80      0.78       340



Few-shot with multiple pics

In [5]:
import os
import json
import base64
import re
import time
import numpy as np
import torch
import pandas as pd
from openai import OpenAI
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import io

os.environ["PYTHONIOENCODING"] = "utf-8"
client = userdata.get('GOOGLE_API_KEY')

# ===============================
# 路径配置
# ===============================
test_image_dir = "/content/drive/MyDrive/MyThesis2026/data/cindy/images/test"
test_csv = "/content/drive/MyDrive/MyThesis2026/data/cindy/data/test.csv"
output_json = "/content/drive/MyDrive/GPT4omini_Misogyny_FewShot_RAG_pred.json"

test_df = pd.read_csv(test_csv)

# ===============================
# 加载 CLIP + 训练集 embeddings
# ===============================
print("Loading CLIP...")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

train_embeddings = np.load("/content/drive/MyDrive/misogyny_train_embeddings.npy")
with open("/content/drive/MyDrive/misogyny_train_meta.json", "r") as f:
    train_meta = json.load(f)

train_filenames = train_meta["filenames"]
train_labels = train_meta["labels"]
print(f"训练集 embeddings 加载完成，共 {len(train_embeddings)} 条")

# ===============================
# RAG 检索函数
# ===============================
def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = clip_model.vision_model(**inputs)
        emb = outputs.pooler_output
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().numpy()

def retrieve_examples(test_emb):
    similarities = train_embeddings @ test_emb
    examples = {}
    for label in [0, 1]:
        label_indices = [i for i, l in enumerate(train_labels) if l == label]
        label_sims = [(i, similarities[i]) for i in label_indices]
        label_sims.sort(key=lambda x: x[1], reverse=True)
        examples[label] = label_sims[0][0]
    return examples

def build_prompt(example_indices):
    label_map = {0: "Non_Misogyny", 1: "Misogyny"}
    examples_text = ""
    for i, (label, idx) in enumerate(example_indices.items()):
        examples_text += f"Example {i+1}: Class label: {label_map[label]}\n"

    return f"""You are an expert in classifying harmful memes in the Chinese language. Your objective is to assess whether a meme is misogynistic or not.

Below are 2 reference examples with their correct labels retrieved from similar memes:

{examples_text}
Now classify the following meme:

Input: [Meme]
Image: [See attached image]
Text embedded: [Read from the image above]

Follow the steps below:
Step 1: Analyze the input meme by assessing both the image and the embedded text, using the provided examples as reference to determine whether it contains misogynistic content.
Step 2: If the meme contains any negative, insulting, stereotyping, or degrading reference to women, output Misogyny.
Step 3: If the meme does not contain any misogynistic content, output Non_Misogyny.

Output:
Your output should strictly follow the format:
Class labels: Misogyny or Non_Misogyny
Thought: Give your reason here"""

def encode_image(image_path):
    image = Image.open(image_path).convert("RGB")
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG")
    buffer.seek(0)
    return base64.b64encode(buffer.read()).decode("utf-8")

def call_with_retry(image_data, prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_data}"}},
                            {"type": "text", "text": prompt}
                        ]
                    }
                ],
                max_tokens=300
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e):
                wait = 60 * (attempt + 1)
                print(f"  Rate limit，等待 {wait} 秒后重试...")
                time.sleep(wait)
            else:
                raise e
    raise Exception("超过最大重试次数")

def parse_label(raw):
    m = re.search(r"Class labels?:\**\s*(Misogyny|Non_Misogyny)", raw, re.IGNORECASE)
    if m:
        return m.group(1)
    elif any(w in raw.lower() for w in ["unable", "cannot", "can't", "sorry"]):
        return "Non_Misogyny"
    elif "misogyn" in raw.lower():
        return "Misogyny"
    else:
        return "UNKNOWN"

# 断点续跑
if os.path.exists(output_json):
    with open(output_json, "r", encoding="utf-8") as f:
        predictions = json.load(f)
    predictions = [p for p in predictions if p["predicted_label"] != "ERROR"]
    done_images = {p["image_name"] for p in predictions}
    print(f"发现已有成功结果 {len(done_images)} 条，从断点继续...")
else:
    predictions = []
    done_images = set()
    print("没有已有结果，从头开始...")

remaining_df = test_df[~test_df["filename"].isin(done_images)]
print(f"剩余待处理: {len(remaining_df)} 张")

for _, row in tqdm(remaining_df.iterrows(), total=len(remaining_df), desc="推理进度"):
    img_name = row["filename"]
    img_path = os.path.join(test_image_dir, img_name)

    if not os.path.exists(img_path):
        print(f"⚠️ 图片不存在: {img_name}")
        continue

    try:
        test_emb = get_embedding(img_path)
        example_indices = retrieve_examples(test_emb)
        prompt_text = build_prompt(example_indices)

        image_data = encode_image(img_path)
        raw = call_with_retry(image_data, prompt_text)
        label = parse_label(raw)

        predictions.append({
            "image_name": img_name,
            "predicted_label": label,
            "raw_output": raw,
            "true_label": int(row["label"])
        })

        print(f"✅ {img_name} -> {label} (真实: {'Misogyny' if row['label']==1 else 'Non_Misogyny'})")

        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)

        time.sleep(3)

    except Exception as e:
        print(f"❌ {img_name} 出错: {e}")
        predictions.append({
            "image_name": img_name,
            "predicted_label": "ERROR",
            "raw_output": str(e),
            "true_label": int(row["label"])
        })
        with open(output_json, "w", encoding="utf-8") as f:
            json.dump(predictions, f, ensure_ascii=False, indent=2)
        time.sleep(5)

print(f"\n完成！共 {len(predictions)} 条结果已保存")
labels = [p["predicted_label"] for p in predictions]
for lbl in sorted(set(labels)):
    print(f"  {lbl}: {labels.count(lbl)}")

Loading CLIP...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

...

✅ 1091.jpg -> Non_Misogyny (真实: Misogyny)


推理进度: 100%|█████████▉| 339/340 [41:19<00:06,  6.69s/it]

✅ 1103.jpg -> Non_Misogyny (真实: Non_Misogyny)


推理进度: 100%|██████████| 340/340 [41:25<00:00,  7.31s/it]


完成！共 340 条结果已保存
  Misogyny: 58
  Non_Misogyny: 282


 Calculation of Few shot

In [6]:
import json
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

output_json = "/content/drive/MyDrive/GPT4omini_Misogyny_FewShot_RAG_pred.json"

with open(output_json, "r", encoding="utf-8") as f:
    predictions = json.load(f)

def normalize_pred(x):
    x = str(x).strip().lower()
    if x == "misogyny":
        return "misogyny"
    return "non_misogyny"

def normalize_true(x):
    return "misogyny" if x == 1 else "non_misogyny"

y_true = [normalize_true(p["true_label"]) for p in predictions]
y_pred = [normalize_pred(p["predicted_label"]) for p in predictions]

print("总数:", len(predictions))
print("Unique TRUE labels:", sorted(set(y_true)))
print("Unique PRED labels:", sorted(set(y_pred)))

ACC = accuracy_score(y_true, y_pred)
MP  = precision_score(y_true, y_pred, average="macro", zero_division=0)
MR  = recall_score(y_true, y_pred, average="macro", zero_division=0)
MF1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
WP  = precision_score(y_true, y_pred, average="weighted", zero_division=0)
WR  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
WF1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print("\nEvaluation Metrics")
print("-------------------------------------------------")
print(f"ACC  : {ACC:.4f}")
print(f"MP   : {MP:.4f}")
print(f"MR   : {MR:.4f}")
print(f"MF1  : {MF1:.4f}")
print(f"WP   : {WP:.4f}")
print(f"WR   : {WR:.4f}")
print(f"WF1  : {WF1:.4f}")
print("-------------------------------------------------")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

总数: 340
Unique TRUE labels: ['misogyny', 'non_misogyny']
Unique PRED labels: ['misogyny', 'non_misogyny']

Evaluation Metrics
-------------------------------------------------
ACC  : 0.7882
MP   : 0.7833
MR   : 0.6888
MF1  : 0.7083
WP   : 0.7862
WR   : 0.7882
WF1  : 0.7676
-------------------------------------------------

Confusion Matrix:
[[ 45  59]
 [ 13 223]]

Classification Report:
              precision    recall  f1-score   support

    misogyny       0.78      0.43      0.56       104
non_misogyny       0.79      0.94      0.86       236

    accuracy                           0.79       340
   macro avg       0.78      0.69      0.71       340
weighted avg       0.79      0.79      0.77       340

